# imports 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np

In [ ]:
pd.options.display.max_rows = 500

# load data

In [ ]:
day = -2
market_data = pd.read_csv(f"./round-1-island-data-bottle/prices_round_1_day_{day}.csv", sep=";", header=0)
trade_history = pd.read_csv(f"./round-1-island-data-bottle/trades_round_1_day_{day}_nn.csv", sep=";", header=0)

In [ ]:
market_data

# research

In [ ]:
starfruit_data = market_data[market_data['product'] == 'STARFRUIT'].reset_index(drop=True)

In [ ]:
starfruit_data['bid_volume_1'].value_counts()

In [ ]:
# Get the value counts of bid_volume_1
bid_volume_counts = starfruit_data['bid_volume_1'].value_counts()

# Create a histogram
plt.figure(figsize=(10, 6))
plt.bar(bid_volume_counts.index, bid_volume_counts.values)
plt.xlabel('Bid Volume 1')
plt.ylabel('Frequency')
plt.title('Histogram of Bid Volume 1')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
columns_to_plot = ['bid_volume_1', 'bid_volume_2', 'bid_volume_3',
                   'ask_volume_1', 'ask_volume_2', 'ask_volume_3']

num_plots = len(columns_to_plot)
num_rows = (num_plots + 1) // 2
num_cols = 2

plt.figure(figsize=(12, 4 * num_rows))

for i, column in enumerate(columns_to_plot, start=1):
    plt.subplot(num_rows, num_cols, i)
    
    volume_counts = starfruit_data[column].value_counts()
    
    plt.bar(volume_counts.index, volume_counts.values)
    plt.xlabel(column)
    plt.ylabel('Frequency')
    plt.title(f'Histogram of {column}')
    plt.xticks(rotation=45)
    plt.tight_layout(pad=3.0)

plt.show()

In [ ]:
def calculate_mm_mid(row):
    # Find the best bid with volume >= 20
    for i in range(1, 4):
        if row[f'bid_volume_{i}'] >= 20:
            best_bid = row[f'bid_price_{i}']
            break
    else:
        best_bid = None

    # Find the best ask with volume >= 20
    for i in range(1, 4):
        if row[f'ask_volume_{i}'] >= 20:
            best_ask = row[f'ask_price_{i}']
            break
    else:
        best_ask = None

    # Calculate the mid price if both best bid and ask are found
    if best_bid is not None and best_ask is not None:
        return (best_bid + best_ask) / 2
    else:
        return None

starfruit_data['mm_mid'] = starfruit_data.apply(calculate_mm_mid, axis=1)

In [ ]:
starfruit_data

In [ ]:
# Create the plot using Plotly Express
fig = px.line(starfruit_data, x='timestamp', y='mm_mid', title='MM Mid Price Over Time')

# Customize the layout
fig.update_layout(
    xaxis_title='Timestamp',
    yaxis_title='MM Mid Price',
)

# Display the plot
fig.show()

In [ ]:
starfruit_fair_prices = starfruit_data[['timestamp', 'mm_mid']]

In [ ]:
starfruit_fair_prices = starfruit_fair_prices.rename(columns={'mm_mid': 'fair'})

In [ ]:
starfruit_fair_prices

In [ ]:
iteration_counts = [1,2,5,10,50, 100, 500,] # we lose too much data with 1000 

In [ ]:
for iterations in iteration_counts:
    starfruit_fair_prices[f"fair_in_{iterations}_its"] = starfruit_fair_prices['fair'].shift(-iterations)
    starfruit_fair_prices[f"fair_{iterations}_its_ago"] = starfruit_fair_prices['fair'].shift(iterations)

In [ ]:
starfruit_fair_prices

In [ ]:
for iterations in iteration_counts:
    starfruit_fair_prices[f'returns_in_{iterations}_its'] = (starfruit_fair_prices[f'fair_in_{iterations}_its'] - starfruit_fair_prices['fair'])/starfruit_fair_prices['fair']
    starfruit_fair_prices[f'returns_from_{iterations}_its_ago'] = (starfruit_fair_prices['fair'] - starfruit_fair_prices[f'fair_{iterations}_its_ago'])/starfruit_fair_prices[f'fair_{iterations}_its_ago']

In [ ]:
starfruit_fair_prices.columns

In [ ]:
starfruit_returns = starfruit_fair_prices[['timestamp','fair','returns_in_1_its', 'returns_from_1_its_ago',
       'returns_in_2_its', 'returns_from_2_its_ago', 'returns_in_5_its',
       'returns_from_5_its_ago', 'returns_in_10_its',
       'returns_from_10_its_ago', 'returns_in_50_its',
       'returns_from_50_its_ago', 'returns_in_100_its',
       'returns_from_100_its_ago', 'returns_in_500_its',
       'returns_from_500_its_ago']]

In [ ]:
starfruit_returns= starfruit_returns.dropna()

In [ ]:
starfruit_returns.mean()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from tqdm import tqdm

# Perform train-test split
train_data, test_data = train_test_split(starfruit_returns, test_size=0.2, random_state=42)

# Iterate over each iteration count
for iterations in tqdm(iteration_counts):
    # Prepare the feature and target columns
    X_train = train_data[f'returns_from_{iterations}_its_ago'].values.reshape(-1, 1)
    y_train = train_data[f'returns_in_{iterations}_its']
    X_test = test_data[f'returns_from_{iterations}_its_ago'].values.reshape(-1, 1)
    y_test = test_data[f'returns_in_{iterations}_its']

    # Create and train the linear regression model
    model = LinearRegression(fit_intercept=False)
    model.fit(X_train, y_train)

    # Make predictions on train and test data
    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)

    # Calculate R-squared and MSE for train and test data
    train_r2 = r2_score(y_train, train_predictions)
    train_mse = mean_squared_error(y_train, train_predictions)
    test_r2 = r2_score(y_test, test_predictions)
    test_mse = mean_squared_error(y_test, test_predictions)

    # Print the results
    print(f"Iteration Count: {iterations}")
    print(f"Learned Equation: returns_in_{iterations}_its = {model.coef_[0]:.4f} * returns_from_{iterations}_its_ago")
    print(f"Train R-squared: {train_r2:.4f}")
    print(f"Train MSE: {train_mse:.4f}")
    print(f"Test R-squared: {test_r2:.4f}")
    print(f"Test MSE: {test_mse:.4f}")
    print()

In [ ]:
!pip install scikit-learn

In [ ]:
1/5000

In [ ]:
-0.2221*1/5000

In [ ]:
_ * 5000